In [15]:
import datetime
import os

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import dates as mdates
from scipy import stats

In [16]:
tick_csv: str = "20260409_9984.csv"
dir_log = "logs_20260409_2"
counter = "000099"

## ティックデータ

In [17]:
df_tick = pd.read_csv(tick_csv)
df_tick.index = [datetime.datetime.fromtimestamp(t) for t in df_tick["Time"]]
df_tick.index.name = "Datetime"
#list_name = ["Time","Price","Volume"]
#tuple(df_tick.iloc[0][list_name])
df_tick

,Time,Price,Volume
Datetime,,,
2026-04-09 09:00:04.222360,1.775693e+09,3823,1359200
2026-04-09 09:00:06.229390,1.775693e+09,3809,1470900
2026-04-09 09:00:08.237190,1.775693e+09,3806,1497100
2026-04-09 09:00:10.243340,1.775693e+09,3810,1681700
2026-04-09 09:00:12.248840,1.775693e+09,3813,1692600
...,...,...,...
2026-04-09 15:24:41.003230,1.775716e+09,3752,38053400
2026-04-09 15:24:43.010250,1.775716e+09,3751,38062900
2026-04-09 15:24:45.002230,1.775716e+09,3752,38070500


## Rewards

In [18]:
log_reward = os.path.join(dir_log, f"reward_{counter}.csv")
df_reward = pd.read_csv(log_reward)
df_reward.sort_values("ts")
df_reward.index = [pd.Timestamp.fromtimestamp(t) for t in df_reward["ts"]]
df_reward["cumsum"] = df_reward["reward"].cumsum()
df_reward = df_reward.sort_index()
df_reward

,ts,reward,cumsum
2026-04-09 09:00:04.222360,1.775693e+09,-1.000,-1.000000
2026-04-09 09:00:06.229390,1.775693e+09,0.210,-0.790000
2026-04-09 09:00:08.237190,1.775693e+09,0.255,-0.535000
2026-04-09 09:00:10.243340,1.775693e+09,0.195,-0.340000
2026-04-09 09:00:12.248840,1.775693e+09,0.150,-0.190000
...,...,...,...
2026-04-09 15:24:39.014330,1.775716e+09,0.030,-426.375991
2026-04-09 15:24:41.003230,1.775716e+09,0.015,-426.360991
2026-04-09 15:24:43.010250,1.775716e+09,0.000,-426.360991
2026-04-09 15:24:45.002230,1.775716e+09,0.015,-426.345991


## Transaction

In [19]:
log_transaction = os.path.join(dir_log, f"transaction_{counter}.csv")
df_transaction = pd.read_csv(log_transaction)
n_contract = len(df_transaction)
df_profit = df_transaction[df_transaction["売買"].isin(["買埋", "売埋"])]
df_profit["累積損益"] = df_profit["損益"].cumsum()
df_profit.index = [pd.to_datetime(s) for s in df_profit["注文日時"]]
df_profit

,注文番号,注文日時,銘柄コード,売買,約定単価,約定数量,損益,備考,累積損益
2026-04-09 09:01:44.455140114,2,2026-04-09 09:01:44.455140114,9984,買埋,3780.0,1,43.0,NaN,43.0
2026-04-09 09:02:26.617719889,4,2026-04-09 09:02:26.617719889,9984,買埋,3770.0,1,11.0,NaN,54.0
2026-04-09 09:02:34.643209934,6,2026-04-09 09:02:34.643209934,9984,買埋,3781.0,1,-6.0,NaN,48.0
2026-04-09 09:02:38.668319941,8,2026-04-09 09:02:38.668319941,9984,買埋,3781.0,1,-1.0,NaN,47.0
2026-04-09 09:02:42.673599958,10,2026-04-09 09:02:42.673599958,9984,売埋,3779.0,1,1.0,NaN,48.0
...,...,...,...,...,...,...,...,...,...
2026-04-09 15:17:50.033580065,218,2026-04-09 15:17:50.033580065,9984,売埋,3749.0,1,4.0,NaN,62.0
2026-04-09 15:18:54.280440092,220,2026-04-09 15:18:54.280440092,9984,売埋,3752.0,1,3.0,NaN,65.0
2026-04-09 15:22:22.762140036,222,2026-04-09 15:22:22.762140036,9984,売埋,3749.0,1,-3.0,NaN,62.0
2026-04-09 15:23:00.835730076,224,2026-04-09 15:23:00.835730076,9984,売埋,3751.0,1,2.0,NaN,64.0


In [20]:
FONT_PATH = "fonts/RictyDiminished-Regular.ttf"
fm.fontManager.addfont(FONT_PATH)

# FontPropertiesオブジェクト生成（名前の取得のため）
font_prop = fm.FontProperties(fname=FONT_PATH)
font_prop.get_name()

plt.rcParams["font.family"] = font_prop.get_name()
plt.rcParams["font.size"] = 9

fig = plt.figure(figsize=(6.8, 6))
ax = dict()
n = 4
gs = fig.add_gridspec(
    n, 1, wspace=0.0, hspace=0.0, height_ratios=[1 if i == 0 else 1 for i in range(n)]
)
for i, axis in enumerate(gs.subplots(sharex="col")):
    ax[i] = axis
    ax[i].grid()

gtitle = f"{tick_csv}, episode={int(counter)}, contract={n_contract:,d}"
ax[0].set_title(gtitle)
ax[0].plot(df_tick["Price"], color="C0")
ax[0].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
ax[0].yaxis.set_major_formatter(ticker.StrMethodFormatter("{x:,.0f}"))
ax[0].set_ylabel("Price")
# ax[0].legend(loc="upper left", fontsize=7)

ax[1].plot(df_reward["reward"], color="C1")
ax[1].set_ylabel("Reward")

ax[2].plot(df_reward["cumsum"], color="C2")
ax[2].set_ylabel("Reward\n(cumsum)")

ax[3].plot(df_profit["累積損益"], color="C3")
ax[3].set_ylabel("累積損益（円/株）")

plt.tight_layout()
img_review = os.path.join(dir_log, f"review_{counter}.png")
print(img_review)
plt.savefig(img_review)
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'fonts/RictyDiminished-Regular.ttf'